# Atelier ML - Apprentissage: Parties 1 et 2
## But principal: choisir le bon Pokemon

**Probleme a resoudre:** apprendre le dresseur le bon Pokemon a utiliser lors d'un combat

Etapes du projet de Machine Learning:
1. Définition du problème à résoudre
2. Acquisition des données d’apprentissages et de tests
3. Préparer et nettoyer les données
4. Analyser, explorer les données
5. Choisir un modèle d’apprentissage
6. Visualiser les résultats, et ajuster ou modifier le modèle d’apprentissage

## Librairies utilisees: 

1. Numpy: Permet la manipulation de tableaux et matrices
2. Pandas: Permet la manipulation et l'analyse de donnees

In [20]:
import numpy as nmp
import pandas as pd

pd.set_option('display.max_columns', None) #Afficher toutes les colonnes
pd.set_option('future.no_silent_downcasting', True) #Supprimer les avertissements de downcasting

/tmp/ipykernel_3399/2650950320.py:5: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  pd.set_option('future.no_silent_downcasting', True) #Supprimer les avertissements de downcasting


# 1. Preparation des donnees
## Les donnees du Pokedex

Les fichiers `pokedex.csv` et `combats.csv` sont situes dans le repertoire `data`.  On utilise `read_csv` pour charger le Pokedex dans un Dataframe.

In [21]:
nosPokemons = pd.read_csv("data/pokedex.csv")

# Affichage des colonnes du DataFrame
print(nosPokemons.columns.values)

<StringArray>
[                 'NUMERO',                     'NOM',
                  'TYPE_1',                  'TYPE_2',
           'POINTS_DE_VIE',          'NIVEAU_ATTAQUE',
          'NIVEAU_DEFENSE', 'NIVEAU_ATTAQUE_SPECIALE',
 'NIVEAU_DEFENSE_SPECIALE',                 'VITESSE',
      'NOMBRE_GENERATIONS',              'LEGENDAIRE']
Length: 12, dtype: str


| Colonne | Signification |
|---|---|
| NUMERO | Identifiant du Pokémon |
| NOM | Nom du Pokémon |
| TYPE_1 | Type primaire |
| TYPE_2 | Type secondaire |
| POINTS_DE_VIE | Points de vie |
| NIVEAU_ATTAQUE | Niveau d'attaque |
| NIVEAU_DEFENSE | Niveau de défense |
| NIVEAU_ATTAQUE_SPECIALE | Niveau d'attaque spéciale |
| NIVEAU_DEFENSE_SPECIALE | Niveau de défense spéciale |
| VITESSE | Vitesse |
| GENERATION | Génération du Pokémon |
| LEGENDAIRE | Pokémon légendaire (booléen) |

In [22]:
# Utilisation de Head du Dataframe pour afficher les dix premieres lignes

print(nosPokemons.head(10))

   NUMERO               NOM TYPE_1  TYPE_2  POINTS_DE_VIE  NIVEAU_ATTAQUE  \
0       1        Bulbizarre  Herbe  Poison             45              49   
1       2        Herbizarre  Herbe  Poison             60              62   
2       3        Florizarre  Herbe  Poison             80              82   
3       4   Mega Florizarre  Herbe  Poison             80             100   
4       5         Salameche    Feu     NaN             39              52   
5       6         Reptincel    Feu     NaN             58              64   
6       7         Dracaufeu    Feu     Vol             78              84   
7       8  Mega Dracaufeu X    Feu  Dragon             78             130   
8       9  Mega Dracaufeu Y    Feu     Vol             78             104   
9      10          Carapuce    Eau     NaN             44              48   

   NIVEAU_DEFENSE  NIVEAU_ATTAQUE_SPECIALE  NIVEAU_DEFENSE_SPECIALE  VITESSE  \
0              49                       65                       65     

Chaque ligne est une **observation** et chaque colonne une **feature**.

## Classement des features
- **Categorisation categorielle** : `TYPE_1`, `TYPE_2`, `LEGENDAIRE`
- **Categorisation numerique discrete** : `NUMERO`, `POINTS_DE_VIE`, `NIVEAU_ATTAQUE`, `NIVEAU_DEFENSE`, `NIVEAU_ATTAQUE_SPECIALE`, `NIVEAU_DEFENSE_SPECIALE`, `VITESSE`
- **Categorisation ordinale** : `GENERATION`

### Feature LEGENDAIRE
Le Machine Learning travaille seulement avec les donnees numeriques, donc "VRAI" et "FAUX" n'est pas exploitable. Il faudra alors transformer VRAI en 1 et FAUX en 0

In [23]:
# replacement de VRAI FAUX par 1 0 
nosPokemonsClean = nosPokemons
nosPokemonsClean['LEGENDAIRE'] = nosPokemonsClean['LEGENDAIRE'].replace("VRAI", 1)
nosPokemonsClean['LEGENDAIRE'] = nosPokemonsClean['LEGENDAIRE'].replace("FAUX", 0)
print(nosPokemonsClean.head())

   NUMERO              NOM TYPE_1  TYPE_2  POINTS_DE_VIE  NIVEAU_ATTAQUE  \
0       1       Bulbizarre  Herbe  Poison             45              49   
1       2       Herbizarre  Herbe  Poison             60              62   
2       3       Florizarre  Herbe  Poison             80              82   
3       4  Mega Florizarre  Herbe  Poison             80             100   
4       5        Salameche    Feu     NaN             39              52   

   NIVEAU_DEFENSE  NIVEAU_ATTAQUE_SPECIALE  NIVEAU_DEFENSE_SPECIALE  VITESSE  \
0              49                       65                       65       45   
1              63                       80                       80       60   
2              83                      100                      100       80   
3             123                      122                      120       80   
4              43                       60                       50       65   

   NOMBRE_GENERATIONS LEGENDAIRE  
0                   1      

### Manque-t-il des donnees ?
On compte les observations et les features avec `shape`, puis on cherche les valeurs absentes avec `info()`.

In [24]:
#Comptage du nombre d'observations et de features
print(nosPokemonsClean.shape)

(800, 12)


In [25]:
#Informations sur notre jeu de données
print(nosPokemonsClean.info())

<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   NUMERO                   800 non-null    int64 
 1   NOM                      799 non-null    str   
 2   TYPE_1                   800 non-null    str   
 3   TYPE_2                   413 non-null    str   
 4   POINTS_DE_VIE            800 non-null    int64 
 5   NIVEAU_ATTAQUE           800 non-null    int64 
 6   NIVEAU_DEFENSE           800 non-null    int64 
 7   NIVEAU_ATTAQUE_SPECIALE  800 non-null    int64 
 8   NIVEAU_DEFENSE_SPECIALE  800 non-null    int64 
 9   VITESSE                  800 non-null    int64 
 10  NOMBRE_GENERATIONS       800 non-null    int64 
 11  LEGENDAIRE               800 non-null    object
dtypes: int64(8), object(1), str(3)
memory usage: 75.1+ KB
None


**Constats :**
- 800 observations et 12 features
- Il manque 1 `NOM` (799 sur 800)
- Il manque 387 `TYPE_2` (413 sur 800). C'est normal : un Pokemon peut ne pas avoir de type secondaire.

### Le nom manquant
On ne peut pas savoir si le nom a une incidence sur le résultat des combats, donc on ne laisse pas l'observation incomplete. On commence par la trouver.

In [26]:
print(nosPokemonsClean[nosPokemonsClean['NOM'].isnull()])

    NUMERO  NOM  TYPE_1 TYPE_2  POINTS_DE_VIE  NIVEAU_ATTAQUE  NIVEAU_DEFENSE  \
62      63  NaN  Combat    NaN             65             105              60   

    NIVEAU_ATTAQUE_SPECIALE  NIVEAU_DEFENSE_SPECIALE  VITESSE  \
62                       60                       70       95   

    NOMBRE_GENERATIONS LEGENDAIRE  
62                   1          0  


Il s'agit de l'observation 62. Les Pokémons sont dans l'ordre du Pokedex national : on regarde le nom du precedent et du suivant, puis on cherche dans le Pokedex ([Pokepedia](https://www.pokepedia.fr/Liste_des_Pok%C3%A9mon_dans_l%27ordre_du_Pok%C3%A9dex_National)).

In [27]:
print(nosPokemonsClean['NOM'][61])
print(nosPokemonsClean['NOM'][63])

Ferosinge
Caninos


Le Pokemon entre Férosinge et Caninos est **Colossinge**. On renseigne la feature `NOM` de l'observation 62.

In [28]:
nosPokemonsClean.loc[62, 'NOM'] = "Colossinge"
print(nosPokemonsClean['NOM'][62])

Colossinge


Cette modification ne touche que le Dataframe, pas le fichier `pokedex.csv` d'origine.